# 10 — AutoML: data in, best reasoner out

By the end of this notebook you will run a complete model-selection
loop on a shared EV-battery corpus, without manually tuning a single
knob. Five stages:

1. **`analyze_corpus`** — six conditional-entropy probes report whether
   the data is well-posed before any training starts.
2. **`auto_select_ssl`** — champion selector across the four SSL modes
   (`laplacian`, `barlow`, `vicreg`, `jepa`).
3. **`sweep`** — GridSearchCV-style cartesian product with a wall-clock
   budget.
4. **`cross_val_score`** — mean ± std of the champion across K folds.
5. **`ensemble_top_k`** — combine the best K configs via Dempster's rule;
   the ensemble inherits calibration.

Every step is seeded via `random_state=42`, so two runs produce
bit-identical numbers. Every score is computed on *held-out infons* —
the graph topology is shared across configs but the teacher signal and
the eval queries only touch the training half.

This is the sklearn bargain applied to graph reasoning: one import,
one loop, the best reasoner wins.


## 1. Build a populated store

We'll use a 12-document EV-battery scenario (the same one the notebooks
08 and 11 use) so the results are directly comparable across the
deep-dive trio.


In [ ]:
import json, os, tempfile
from cognition import Cognition, CognitionConfig

SCHEMA = {
    "toyota":   {"type": "actor",    "tokens": ["toyota"]},
    "honda":    {"type": "actor",    "tokens": ["honda"]},
    "tesla":    {"type": "actor",    "tokens": ["tesla"]},
    "panasonic":{"type": "actor",    "tokens": ["panasonic"]},
    "catl":     {"type": "actor",    "tokens": ["catl"]},
    "invests":  {"type": "relation", "tokens": ["invest", "invests", "investment"]},
    "partners": {"type": "relation", "tokens": ["partner", "partners", "partnership"]},
    "produces": {"type": "relation", "tokens": ["produce", "produces", "produced"]},
    "expands":  {"type": "relation", "tokens": ["expand", "expands", "expansion"]},
    "delays":   {"type": "relation", "tokens": ["delay", "delays", "delayed"]},
    "acquires": {"type": "relation", "tokens": ["acquire", "acquires", "acquired"]},
    "battery":  {"type": "feature",  "tokens": ["battery", "batteries"]},
    "factory":  {"type": "feature",  "tokens": ["factory", "plant"]},
    "ev":       {"type": "feature",  "tokens": ["ev", "electric vehicle"]},
    "japan":    {"type": "market",   "tokens": ["japan", "japanese"]},
    "china":    {"type": "market",   "tokens": ["china", "chinese"]},
    "na":       {"type": "market",   "tokens": ["north america", "united states"]},
}

DOCS = [
    {"id": "d01", "timestamp": "2024-01-10", "text": "Toyota invests in battery technology in Japan."},
    {"id": "d02", "timestamp": "2024-02-15", "text": "Toyota partners with Panasonic on battery development."},
    {"id": "d03", "timestamp": "2024-03-22", "text": "Toyota produces prototype batteries."},
    {"id": "d04", "timestamp": "2024-01-05", "text": "Tesla expands its battery factory in North America."},
    {"id": "d05", "timestamp": "2024-02-28", "text": "Tesla produces batteries at its Gigafactory."},
    {"id": "d06", "timestamp": "2024-04-01", "text": "Tesla acquires battery supply chain assets."},
    {"id": "d07", "timestamp": "2024-01-20", "text": "Honda partners with CATL on battery supply in China."},
    {"id": "d08", "timestamp": "2024-02-10", "text": "Honda delays its EV production timeline."},
    {"id": "d09", "timestamp": "2024-03-15", "text": "Honda invests in battery research."},
    {"id": "d10", "timestamp": "2024-01-25", "text": "CATL expands battery production in China."},
    {"id": "d11", "timestamp": "2024-02-20", "text": "CATL produces batteries for Japanese automakers."},
    {"id": "d12", "timestamp": "2024-01-30", "text": "Panasonic invests in battery factory in Japan."},
]

tmpdir = tempfile.mkdtemp(prefix="automl-")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

cog = Cognition(CognitionConfig(
    schema_path=schema_path,
    db_path=os.path.join(tmpdir, "cog.db"),
    quality_threshold=0.05,
    max_triples_per_sentence=2,
    random_state=42,
))
for d in DOCS:
    cog.ingest([d])
cog.consolidate()
print(f"ingested {len(DOCS)} docs → {cog.stats()['infon_count']} infons")


## 2. Diagnose before training

`analyze_corpus(cog)` walks the store once and reports:

| Probe | What it catches |
|---|---|
| `H(O \| S, P)` | deterministic (memorisation) vs pure noise |
| hub concentration | one anchor dominating; contrastive scoring collapses |
| triple coverage | saturation — no room for genuine NEI queries |
| temporal edge density | not enough NEXT edges for Temporal-JEPA |
| relational asymmetry | whether sheaf's `P_fwd ≠ P_bwd` has anything to exploit |
| role-marginal entropy | role imbalance |

Warnings fire when a measurement crosses a known-problematic threshold.
Run this before committing compute — it's free.


In [ ]:
from cognition.diagnostics import analyze_corpus

report = analyze_corpus(cog)
print(report.summary())


A 50% hub concentration on `battery` is the exact pattern we saw kill
the latent planner in `examples/planner_scale.py`. For this corpus
it's fine — we're warned, we move on — but on a production corpus this
would be where you decide to split `battery` into `battery_tech`,
`battery_pack`, `battery_supply` before training.


## 3. `auto_select_ssl` — the champion across pretext losses

Four SSL modes, each fit on the same held-out infon split, scored on a
self-generated NEI/SUPPORTS query set derived from withheld infons.
`ssl_mode='laplacian'` is the default (sheaf-Laplacian regularizer);
the others are opt-in pretext losses we shipped via `cognition.ssl`.


In [ ]:
from cognition.model_selection import auto_select_ssl

champions = auto_select_ssl(
    cog,
    modes=["laplacian", "barlow", "vicreg", "jepa"],
    heldout_frac=0.3,
    epochs=10,
    verbose=True,
)

print()
print(f"{'mode':<12s} {'score':>8s} {'n_heldout':>10s}")
print("-" * 32)
for r in champions:
    print(f"{r.config_overrides['ssl_mode']:<12s} "
          f"{r.score:>7.0%} "
          f"{r.n_heldout:>10d}")


Ranked champion at index 0. On a tiny corpus like this one, the
differences between modes are small because the *relevance filter*
caps accuracy — see `11_calibration_and_theta.ipynb` for why. On a
larger corpus, BT and JEPA start to pull ahead as rank-expansion cashes
in. See `planner_scale.py` in `examples/` for that behaviour.


## 4. `sweep` with a wall-clock budget

Grid search across `ssl_mode × hidden_dim`. The `time_budget_sec`
parameter stops launching new configs once the wall clock exceeds the
budget — essential for big grids.


In [ ]:
from cognition.model_selection import sweep

results = sweep(
    cog,
    grid={
        "ssl_mode":   ["laplacian", "barlow"],
        "hidden_dim": [16, 32],
    },
    heldout_frac=0.3,
    epochs=10,
    time_budget_sec=180,   # generous — this grid is 4 configs
    verbose=True,
)

print()
print(f"{'ssl_mode':<12s} {'hidden':>8s} {'score':>8s}")
print("-" * 32)
for r in results:
    print(f"{r.config_overrides['ssl_mode']:<12s} "
          f"{r.config_overrides['hidden_dim']:>8d} "
          f"{r.score:>7.0%}")


The top result is our champion candidate. Before trusting it, measure
how reliably it wins — one held-out split is vulnerable to luck.


## 5. `cross_val_score` — mean ± std across folds

K-fold rotates the held-out set so every infon takes a turn as eval.
The resulting mean±std tells you whether the champion is *reliably*
the champion or just won one lottery.


In [ ]:
from cognition.model_selection import cross_val_score

champion = results[0].config_overrides
cv = cross_val_score(cog, config_overrides=champion, k=3, epochs=8, verbose=True)

print()
print(f"champion:    {champion}")
print(f"mean score:  {cv['mean_score']:.0%}")
print(f"std  score:  {cv['std_score']:.0%}")
print(f"folds:       {cv['per_fold']}")


A tight std means the champion is robust across folds. A wide std
means you got lucky once and should sweep more broadly.


## 6. `ensemble_top_k` — combine via Dempster's rule

The top-K configs can be ensembled by combining their per-query mass
functions with Dempster's rule. The key property: **the ensemble
inherits DS calibration**. If all K members were ignorant on a claim,
the ensemble is too — θ stays high. If one is confident and the others
are ignorant, the ensemble inherits the confident one's mass.

Most ensemble schemes (averaging, voting) would silently *erase* the θ
channel. Dempster's rule doesn't.


In [ ]:
from cognition.model_selection import ensemble_top_k

ens = ensemble_top_k(
    cog,
    grid={"ssl_mode": ["laplacian", "barlow"], "hidden_dim": [16, 32]},
    k=3,
    heldout_frac=0.3,
    epochs=10,
    verbose=True,
)

print()
print(f"ensemble score:     {ens['ensemble_score']:.0%}")
print(f"per-member scores:  "
      f"{[round(s, 2) for s in ens['per_member_scores']]}")
print()
print(f"first few ensemble verdicts:")
for i, (q, v, m) in enumerate(zip(ens['queries'],
                                    ens['ensemble_verdicts'],
                                    ens['ensemble'])):
    if i >= 5: break
    print(f"  {v:<18s}  θ={m.theta:.2f}   "
          f"{q['q']}")


## 7. The fitted champion, ready to use

Everything we've done is model-selection *over the store*. Once you've
picked the champion, `cog.reasoner(**champion)` returns a fit reasoner
you can drop into any agent pipeline. The AutoML loop's output is a
*reasoner*, not a configuration file.


In [ ]:
reasoner = cog.reasoner()

# One worked query showing the reasoner output
r = reasoner.reason("Did Toyota invest in batteries?")
print(f"verdict:  {r.verdict}")
print(f"supports: {r.mass.supports:.3f}")
print(f"theta:    {r.mass.theta:.3f}")
print()

# And the sklearn-style contract for batch use
preds = reasoner.predict([
    "Did Toyota invest in batteries?",
    "Did Tesla acquire CATL?",                       # NEI
    "Did Honda partner with CATL?",
])
for q, v in zip(["Toyota/invest/battery",
                 "Tesla/acquire/CATL",
                 "Honda/partner/CATL"], preds):
    print(f"  {q:<28s} → {v}")


## Recap

- `analyze_corpus` told us the data was workable (50% hub on
  `battery` noted, proceed anyway).
- `auto_select_ssl` ranked the four pretext losses on held-out infons;
  `laplacian` was the champion on this corpus.
- `sweep` expanded to `ssl_mode × hidden_dim` with a wall-clock budget.
- `cross_val_score` estimated variance across 3 folds.
- `ensemble_top_k` combined the top-3 via Dempster's rule — the
  ensemble preserves calibration (θ high on NEI, committed on SUPPORTS).

**No knob was tuned by hand.** The only configuration input was
`random_state=42`. Running this notebook with a different seed will
produce a different champion; running it again with the same seed will
produce bit-identical numbers.

**Where to go next:**
- **`11_calibration_and_theta.ipynb`** — *why* `θ` is load-bearing and
  how the relevance filter earned it.
- **`09_self_supervised_mdp.ipynb`** — the research arc behind the
  SSL modes this notebook picked between.
- `cognition-workshop/14-automl-loop.md` — the prose module that
  builds every helper used above from first principles.


In [ ]:
cog.close()
